In [2]:
import torch
from sklearn.model_selection import train_test_split
from torch import nn
import os 
import csv
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from sklearn.metrics import confusion_matrix


from src.models import MultiClassifierV2_STN, CVAE
from src.utils import process_spectra_data, process_spectra_csv_data

# Device Setup

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Optionally, you can print more details about the GPU if available
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs available: {torch.cuda.device_count()}")

Using device: cuda
GPU Name: Tesla T4
Number of GPUs available: 1


## Single Model Training


In [ ]:
from src.utils import  preprocess_data_for_classifier, train_model
from src.models import MultiClassifierV2_STN

max_shift = 0 # in eV
max_broadening_sigma = 0.5


train_loader, test_loader, input_features, output_features = preprocess_data_for_classifier(
    spectra_dir= '../data/synthetic_data_hard_final/synthetic_spectra',
    labels_dir='../data/synthetic_data_hard_final/synthetic_labels',
    max_shift=max_shift,
    max_broadening_sigma=max_broadening_sigma,
    valence_range=0,  #valence region is already removed in saved data
    energy_resolution=0.1, 
    test_size=0.2,
    random_state=42,
    batch_size=64)
# 4. Create an instance of the model and send it to target device
model_0 =  MultiClassifierV2_STN().to(device)
#Pielsticker_CNN().to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model_0.parameters(), lr=1e-4)

# Calculate accuracy (a classification metric)
def accuracy_fn(y_true, y_pred):
    # Check if all 39 classes match for each sample
    correct = (y_true == y_pred).all(axis=1).sum().item()  # Use .all(axis=1) to check all classes
    acc = (correct / len(y_true)) * 100  # Calculate accuracy as a percentage
    return acc

# Create a list to store loss values
training_loss_values = []
testing_loss_values = []


# Train the model
training_information = train_model(
    model=model_0,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device,
    epochs=1000,
    print_freq=1 # Print every 100 epochs
)

training_loss_values = training_information['train_loss']
testing_loss_values = training_information['test_loss']
trained_model = training_information['model']

Epoch:    0 | Train Loss: 0.25005, Acc: 15.55% | Test Loss: 0.12731, Acc: 23.10%
Epoch:    1 | Train Loss: 0.10652, Acc: 26.97% | Test Loss: 0.08811, Acc: 34.95%
Epoch:    2 | Train Loss: 0.08292, Acc: 35.99% | Test Loss: 0.07074, Acc: 42.84%
Epoch:    3 | Train Loss: 0.06886, Acc: 43.52% | Test Loss: 0.05728, Acc: 51.94%
Epoch:    4 | Train Loss: 0.05940, Acc: 49.47% | Test Loss: 0.04934, Acc: 57.67%
Epoch:    5 | Train Loss: 0.05317, Acc: 53.68% | Test Loss: 0.04171, Acc: 63.20%
Epoch:    6 | Train Loss: 0.04935, Acc: 56.56% | Test Loss: 0.04090, Acc: 63.02%
Epoch:    7 | Train Loss: 0.04626, Acc: 58.67% | Test Loss: 0.03723, Acc: 66.06%
Epoch:    8 | Train Loss: 0.04418, Acc: 60.32% | Test Loss: 0.03544, Acc: 67.12%
Epoch:    9 | Train Loss: 0.04268, Acc: 61.50% | Test Loss: 0.03639, Acc: 66.45%
Epoch:   10 | Train Loss: 0.04088, Acc: 62.90% | Test Loss: 0.03348, Acc: 68.54%
Epoch:   11 | Train Loss: 0.04027, Acc: 63.37% | Test Loss: 0.03202, Acc: 70.07%
Epoch:   12 | Train Loss: 0.

KeyboardInterrupt: 

# Training looper

this loops performs training for models on data at different levels of shift applied. This provides a good point of comparison for how different models can handle horizontal shifts 

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
import os
from tqdm.notebook import tqdm
import json
from src.utils import process_spectra_data, train_model, compute_class_metrics, get_label_dict 

# ==============================================================================
# 🚀 SECTION 1: HELPER FUNCTIONS & CLASSES
# (Moved from previous cells to make this script self-contained)
# ==============================================================================

class FocalLoss(nn.Module):
    """
    Focal Loss, a modification of BCE Loss for class imbalance.
    """
    def __init__(self, gamma=2, alpha=0.25):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
    
    def forward(self, inputs, targets):
        BCE_loss = nn.BCEWithLogitsLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-BCE_loss)  # Probabilities for the correct class
        F_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss  # Focal loss term
        return F_loss.mean()

def accuracy_fn(y_true, y_pred):
    """
    Calculates exact match accuracy for multi-label classification.
    """
    # Check if all classes match for each sample
    correct = (y_true == y_pred).all(axis=1).sum().item()
    acc = (correct / len(y_true)) * 100  # Accuracy as a percentage
    return acc

def init_weights(m):
    """
    Applies Kaiming Normal initialization to Conv1d and Linear layers.
    """
    if isinstance(m, nn.Conv1d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)



# ==============================================================================
# ⚙️ SECTION 2: CONFIGURATION
# ==============================================================================

# --- Experiment Setup ---
model_name = 'STN_Classifier_Final' # A descriptive name for this run
# Select the model architecture you want to use
selected_model_class = MultiClassifierV2_STN 
# Define the output directory for all reports and results
report_dir = Path(f'../reports/final_results/{model_name}_results')

# --- Hyperparameters ---
# You can choose which loss function to use here
loss_fn = nn.BCEWithLogitsLoss() 
# loss_fn = FocalLoss() # <-- Or uncomment this to use Focal Loss

LEARNING_RATE = 1e-4
EPOCHS = 1000
PRINT_FREQ = 1 # Print progress every 100 epochs

# --- Data Parameters ---
# Define the shift values (in eV) you want to loop over
shift_values = [0.0, 1.0, 2.0, 3.0] 
max_broadening_sigma = 0.01

# --- Get the list of Functional Group names (ensure FG_list is defined in your utils) ---
with open("../data/FG_list.json", "r") as f:
    FG_list = json.load(f)

# ==============================================================================
# 🚀 SECTION 3: MAIN EXPERIMENT WORKFLOW
# ==============================================================================

# Create the main report directory
report_dir.mkdir(parents=True, exist_ok=True)
# Initialize a list to store the summary results from each run
all_run_summaries = []
# Set the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# --- Main loop over different experimental conditions (shifts) ---
for max_shift in shift_values:
    print(f"\n{'='*60}")
    print(f"STARTING EXPERIMENT: max_shift = {max_shift} eV")
    print(f"{'='*60}")
    
    # 1. Process data for the current shift value
    print("Loading and processing data...")
    train_loader, test_loader, _, _ = process_spectra_data(
        spectra_dir= '../data/synthetic_data_hard_final/synthetic_spectra',
        labels_dir='../data/synthetic_data_hard_final/synthetic_labels',
        max_shift=max_shift,
        max_broadening_sigma=max_broadening_sigma,
        valence_range=0, # Assumes valence region is already removed
        energy_resolution=0.1,
        test_size=0.2,
        random_state=42,
        batch_size=64
    )
    
    # 2. Initialize model and optimizer for this run
    print("Initializing model...")
    model = selected_model_class().to(device)
    model.apply(init_weights) # Apply weight initialization
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # 3. Train the model
    print("Starting model training...")
    training_info = train_model(
        model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device,
        epochs=EPOCHS,
        print_freq=PRINT_FREQ
    )


Using device: cuda

STARTING EXPERIMENT: max_shift = 0.0 eV
Loading and processing data...
Initializing model...
Starting model training...
Epoch:    0 | Train Loss: 0.30362, Acc: 0.71% | Test Loss: 0.62198, Acc: 2.44%
Epoch:    1 | Train Loss: 0.21245, Acc: 0.81% | Test Loss: 0.51351, Acc: 0.84%
Epoch:    2 | Train Loss: 0.20941, Acc: 0.78% | Test Loss: 0.67498, Acc: 0.25%
Epoch:    3 | Train Loss: 0.20813, Acc: 0.75% | Test Loss: 0.75883, Acc: 1.05%
Epoch:    4 | Train Loss: 0.20749, Acc: 0.76% | Test Loss: 0.88050, Acc: 1.05%
Epoch:    5 | Train Loss: 0.20698, Acc: 0.75% | Test Loss: 0.59747, Acc: 0.30%


KeyboardInterrupt: 

# Save the Model

In [ ]:
# Save the model
model_path = Path("/home/issa/new_project_clone/local_models") / f"best_STN_{max_shift}_shift.pth"
torch.save(model_0.state_dict(), model_path)
print(f"Model saved to {model_path}")


Model saved to /home/issa/new_project_clone/local_models/best_STN_5_shift.pth


## Load Saved Model

In [ ]:
# Create an instance of your model
saved_model = MultiClassifierV1_STN().to(device)

# Define the path to the saved model
model_path = Path("/home/issa/new_project_clone/local_models") / f"STN_{max_shift}_shift.pth"

# Load the state dictionary
saved_model.load_state_dict(torch.load(model_path))

# Set the model to evaluation mode (if you're going to use it for inference)
saved_model.eval()

print(f"Model loaded from {model_path}")

Model loaded from /home/issa/new_project_clone/local_models/STN_3_shift.pth


## Testing

In [ ]:

def print_class_confusion_matrices(y_true, y_pred, class_names=None):
    """
    Print normalized confusion matrices for each class individually,
    handling cases where classes might be empty.
    
    Args:
        y_true: True labels (n_samples × n_classes)
        y_pred: Predicted labels (n_samples × n_classes)
        class_names: Optional list of class names
    """
    n_classes = y_true.shape[1]
    if class_names is None:
        class_names = [f"Class {i}" for i in range(n_classes)]
    
    # Initialize DataFrame to store performance metrics
    metrics_df = pd.DataFrame(columns=['Class', 'TP', 'TN', 'FP', 'FN', 
                                     'Precision', 'Recall', 'F1', 'Support'])
    
    for i in range(n_classes):
        # Ensure we always consider both classes (0 and 1) even if empty
        cm = confusion_matrix(y_true[:, i], y_pred[:, i], labels=[0, 1])
        
        # Calculate metrics
        tn, fp, fn, tp = cm.ravel()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        support = tp + fn
        
        # Store metrics
        metrics_df.loc[i] = [
            class_names[i],
            tp, tn, fp, fn,
            f"{precision:.3f}",
            f"{recall:.3f}",
            f"{f1:.3f}",
            support
        ]
        
        # Normalize by true labels (rows)
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
        
        print(f"\n{class_names[i]} Confusion Matrix (Normalized by True Labels):")
        print(f"Actual\Predicted | Negative | Positive")
        print("-"*40)
        print(f"Negative         | {cm_normalized[0,0]:5.1f}%   | {cm_normalized[0,1]:5.1f}%")
        print(f"Positive         | {cm_normalized[1,0]:5.1f}%   | {cm_normalized[1,1]:5.1f}%")
    
    # Print summary metrics
    print("\n\nClassification Metrics Summary:")
    print(metrics_df.to_string(index=False))
    
    return metrics_df

<>:44: SyntaxWarning: invalid escape sequence '\P'
<>:44: SyntaxWarning: invalid escape sequence '\P'
/tmp/ipykernel_2562669/566775622.py:44: SyntaxWarning: invalid escape sequence '\P'
  print(f"Actual\Predicted | Negative | Positive")


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

metrics = print_class_confusion_matrices(
    y_true=test_labels_np,
    y_pred=test_preds_np,
    class_names= FG_list  # Custom names if available
)


alkene Confusion Matrix (Normalized by True Labels):
Actual\Predicted | Negative | Positive
----------------------------------------
Negative         |  99.6%   |   0.4%
Positive         |  22.8%   |  77.2%

alkyne Confusion Matrix (Normalized by True Labels):
Actual\Predicted | Negative | Positive
----------------------------------------
Negative         | 100.0%   |   0.0%
Positive         |   nan%   |   nan%

benzene ring Confusion Matrix (Normalized by True Labels):
Actual\Predicted | Negative | Positive
----------------------------------------
Negative         |  99.0%   |   1.0%
Positive         |   4.3%   |  95.7%

naphthalene Confusion Matrix (Normalized by True Labels):
Actual\Predicted | Negative | Positive
----------------------------------------
Negative         | 100.0%   |   0.0%
Positive         |  12.2%   |  87.8%

amine Confusion Matrix (Normalized by True Labels):
Actual\Predicted | Negative | Positive
----------------------------------------
Negative         |  99.9

/tmp/ipykernel_2562669/566775622.py:41: RuntimeWarning: invalid value encountered in divide
  cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
/tmp/ipykernel_2562669/566775622.py:41: RuntimeWarning: invalid value encountered in divide
  cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
/tmp/ipykernel_2562669/566775622.py:41: RuntimeWarning: invalid value encountered in divide
  cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
/tmp/ipykernel_2562669/566775622.py:41: RuntimeWarning: invalid value encountered in divide
  cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
/tmp/ipykernel_2562669/566775622.py:41: RuntimeWarning: invalid value encountered in divide
  cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
/tmp/ipykernel_2562669/566775622.py:41: RuntimeWarning: invalid value encountered in divide
  cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 10

# Testing

In [ ]:
# 4. Create an instance of the model and send it to target device
model_0 = MultiClassifierV1().to(device)

# Load the saved model
model_path = Path("../models/multi_class_models/no_shift_v0.pth")
model_0.load_state_dict(torch.load(model_path))
model_0.eval()

X_test = X_test.to(device)

# Generate predictions using the loaded model
with torch.inference_mode():
    test_logits = model_0(X_test).squeeze() 
    test_pred = torch.round(torch.sigmoid(test_logits))


FileNotFoundError: [Errno 2] No such file or directory: '../models/multi_class_models/no_shift_v0.pth'

In [ ]:
model_0.eval()

X_test = X_test.to(device)

# Generate predictions using the loaded model
with torch.inference_mode():
    test_logits = model_0(X_test).squeeze() 
    test_pred = torch.round(torch.sigmoid(test_logits))

In [ ]:
from sklearn.metrics import recall_score
import numpy as np
import os
import pandas as pd 
from tabulate import tabulate

def get_label_dict(path_to_file):
    '''
    Extract the list of functional groups
    '''
    spreadsheet_f = pd.ExcelFile(path_to_file)
    df_f = pd.read_excel(spreadsheet_f)
    return list(df_f['Functional groups'])

data_path = Path("../data/experimental_data")
CEL_FG_path = os.path.join(data_path, 'cellulose (CEL)', 'CEL_FG.xlsx')
FG_list = get_label_dict(CEL_FG_path)
FG_list.append('alkane')

y_test_np = y_test.cpu().numpy()
test_pred_np = test_pred.cpu().numpy()

report_array = []  # New array for sensitivity values

# Calculate sensitivity for each class and total frequency
for i in range(39):  # Assuming 39 classes
    # Check if the class has positive samples in the test data
    if np.sum(y_test_np[:, i]) > 0:  # If there are positive samples
        sensitivity = recall_score(y_test_np[:, i], test_pred_np[:, i], average=None)
        total_frequency = np.sum(bin_label_array[:, i])  # Count total occurrences of the class in the training data
        report_array.append([i, FG_list[i], sensitivity[1], total_frequency])  # Append class index, sensitivity value, total frequency, and functional group
    else:
        report_array.append([i, FG_list[i], None, 0])  # No sensitivity value for this class, frequency is 0, and functional group

# Calculate the average sensitivity value, ignoring empty entries
sensitivity_values = [row[2] for row in report_array if row[2] is not None]
average_sensitivity = np.mean(sensitivity_values)
    

print(tabulate(report_array, headers=['Class No.', 'Functional Group', 'Sensitivity', 'Total Frequency', ]))
print(f"\nAverage sensitivity (excluding classes with no data): {average_sensitivity:.2f}")

pd.DataFrame(report_array, columns=['Class No.', 'Functional Group', 'Sensitivity', 'Total Frequency']).to_csv('../reports/model0.csv', index=False)





  Class No.  Functional Group       Sensitivity    Total Frequency
-----------  -------------------  -------------  -----------------
          0  alkene                    0.856716               1739
          1  alkyne                                              0
          2  benzene ring              0.96881                8039
          3  naphthalene               0.75                    361
          4  amine                     0.949533               2569
          5  alcohol (aromatic)        0.793103                375
          6  alcohol (aliphatic)       0.840125               1448
          7  ether (aromatic)          0.938017               2471
          8  ether (aliphatic)         0.953913               5476
          9  alkyl halide (F)          0.989362               1882
         10  alkyl halide (Cl)         0.988764               1405
         11  alkyl halide (Br)                                   0
         12  alkyl halide (I)                                 